# Microsoft Agent Framework — Azure OpenAI (API Απαντήσεων)

Σε αυτό το δείγμα κώδικα, θα χρησιμοποιήσετε το **Microsoft Agent Framework (MAF)** για να δημιουργήσετε έναν απλό πράκτορα που υποστηρίζεται από **Azure OpenAI** χρησιμοποιώντας το **API Απαντήσεων**.

> **Σημείωση μετανάστευσης:** Αυτό το δείγμα χρησιμοποιούσε προηγουμένως το Semantic Kernel με GitHub Models. Έχει μεταφερθεί στο Microsoft Agent Framework, και τα GitHub Models (καταργούνται, θα αποσυρθούν τον Ιούλιο του 2026) έχουν αντικατασταθεί με Azure OpenAI, που υποστηρίζει το API Απαντήσεων. Ο `OpenAIChatClient` στο MAF στοχεύει στο σταθερό σημείο τερματισμού `/openai/v1/` του Azure OpenAI και χρησιμοποιεί το API Απαντήσεων ως προεπιλογή.

Ο σκοπός αυτού του δείγματος είναι να δείξει τα βήματα που θα εφαρμοστούν αργότερα σε επιπλέον δείγματα κώδικα κατά την υλοποίηση διάφορων προτύπων πρακτόρων.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Εισαγωγή των Απαραίτητων Πακέτων Python


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Ορισμός ενός Εργαλείου

Στο Microsoft Agent Framework, ένα **εργαλείο** είναι μια απλή συνάρτηση Python διακοσμημένη με `@tool` την οποία μπορεί να καλέσει ο agent. Παρακάτω ορίζουμε ένα εργαλείο που επιστρέφει έναν τυχαίο προορισμό διακοπών και αποφεύγει να επαναλάβει τον προηγούμενο.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Δημιουργία του Αντιπροσώπου

Εδώ, δημιουργούμε τον Αντιπρόσωπο με όνομα `TravelAgent`.

Σε αυτό το παράδειγμα, χρησιμοποιούμε πολύ βασικές οδηγίες. Μη διστάσετε να τροποποιήσετε αυτές τις οδηγίες για να δείτε πώς αλλάζει η συμπεριφορά του αντιπροσώπου.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Εκτέλεση του Agent

Τώρα μπορούμε να εκτελέσουμε τον agent. Δημιουργούμε μια `AgentSession` ώστε ο agent να θυμάται τη συνομιλία ανάμεσα στις στροφές, και στη συνέχεια στέλνουμε δύο `user_inputs`. Το πρώτο ζητά ένα ταξίδι· το δεύτερο λέει ότι στον χρήστη δεν άρεσε η πρόταση και ζητά άλλη — ο agent χρησιμοποιεί το ιστορικό της συνεδρίας μαζί με το εργαλείο `get_random_destination` για να απαντήσει.

Μπορείτε να τροποποιήσετε αυτά τα μηνύματα για να παρατηρήσετε πώς αντιδρά διαφορετικά ο agent. Οι απαντήσεις **εκπέμπονται** διαδοχικά token-ανά-token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
